# 06 — Fine-tuning QLoRA del LLM especializado en química

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Fine-tuning — Paso 2 de 4

---

Este notebook entrena un LLM especializado en química organometálica
mediante QLoRA sobre el dataset construido en `05_dataset.ipynb`.

El output de este notebook es el modelo fine-tuneado publicado en HuggingFace Hub.

In [1]:
"""
Notebook: 06_finetune.ipynb

Objetivo:
    Fine-tuning de Qwen2.5-1.5B-Instruct sobre el dataset de química organometálica
    construido en 05_dataset.ipynb mediante QLoRA (Quantized Low-Rank Adaptation).

    QLoRA combina dos técnicas:
          - Quantization (4-bit): reduce el modelo base a 4 bits para que quepa en
            la GPU T4 gratuita de Colab (15GB VRAM)
          - LoRA (Low-Rank Adaptation): añade matrices de adaptación pequeñas sobre
            las capas del modelo y entrena solo esas matrices (~1-3% de parámetros)
            dejando los pesos originales del modelo base intactos

    A diferencia del pipeline RAG donde Qwen3-8B genera respuestas usando
    contexto externo sin modificar sus pesos, aquí modificamos los pesos
    de los adaptadores LoRA para que el modelo internalice el dominio
    de la química organometálica directamente.

    Nota sobre la elección del modelo base:
          Se usa Qwen2.5-1.5B-Instruct en lugar de Qwen3-8B (usado en el RAG)
          por restricciones de VRAM en la T4 gratuita (15GB). Usar modelos de
          tamaño diferente para inferencia en producción y fine-tuning es la
          práctica estándar en la industria — cada componente se optimiza para
          su función concreta.

Fuente de datos:
    Dataset: Jesusrodriguezf90/chemistry-organometallic-qa (HF Hub)
    73 pares QA sobre el paper PMC10967698 — Büchele et al. (2024).

Siguiente paso:
    06b_evaluacion.ipynb — Evaluación comparativa base vs fine-tuned

Autor:   Jesús Rodríguez
Fecha:   2026-05-19
Versión: 1.0.0
"""

'\nNotebook: 06_finetune.ipynb\n\nObjetivo:\n    Fine-tuning de Qwen2.5-1.5B-Instruct sobre el dataset de química organometálica\n    construido en 05_dataset.ipynb mediante QLoRA (Quantized Low-Rank Adaptation).\n    \n    QLoRA combina dos técnicas:\n          - Quantization (4-bit): reduce el modelo base a 4 bits para que quepa en\n            la GPU T4 gratuita de Colab (15GB VRAM)      \n          - LoRA (Low-Rank Adaptation): añade matrices de adaptación pequeñas sobre\n            las capas del modelo y entrena solo esas matrices (~1-3% de parámetros)\n            dejando los pesos originales del modelo base intactos\n\n    A diferencia del pipeline RAG donde Qwen3-8B genera respuestas usando\n    contexto externo sin modificar sus pesos, aquí modificamos los pesos\n    de los adaptadores LoRA para que el modelo internalice el dominio\n    de la química organometálica directamente.\n    \n    Nota sobre la elección del modelo base:\n          Se usa Qwen2.5-1.5B-Instruct en luga

## 1. Configuración del entorno

In [2]:
# Librería estándar
import json
import os
from pathlib import Path

# Third-party — se importan tras la instalación en Sección 2
# Ver celdas de instalación antes de ejecutar los imports siguientes
import sys
print(f'Python version: {sys.version}')

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
# Verificar disponibilidad de GPU — requisito para fine-tuning QLoRA
# La T4 de Colab gratuita tiene 15GB VRAM — suficiente para Qwen2.5-1.5B en 4-bit
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU detectada: {result.stdout.strip()}')
else:
    raise RuntimeError(
        'GPU no detectada. Activa la GPU en Colab: '
        'Runtime → Change runtime type → T4 GPU'
    )

GPU detectada: Tesla T4, 15360 MiB


In [4]:
from google.colab import drive, userdata
drive.mount('/content/drive')

PROYECTO_RAIZ = Path('/content/drive/MyDrive/chem-rag-assistant')
DIR_FINETUNE  = PROYECTO_RAIZ / 'data' / 'finetune'
DIR_FINETUNE.mkdir(parents=True, exist_ok=True)

print(f'Proyecto raíz: {PROYECTO_RAIZ}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto raíz: /content/drive/MyDrive/chem-rag-assistant


In [5]:
# Cargar token HF desde los secretos de Colab
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF_TOKEN cargado desde secretos de Colab.')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')
    print('HF_TOKEN cargado desde variable de entorno.')

assert HF_TOKEN, 'HF_TOKEN no encontrado. Añádelo en Colab → Secrets.'

# Configuración del modelo y dataset
MODELO_BASE   = 'Qwen/Qwen2.5-1.5B-Instruct'
HF_USERNAME   = 'Jesusrodriguezf90'
NOMBRE_MODELO = 'qwen2.5-1.5b-chemistry-lora'
DATASET_ID    = f'{HF_USERNAME}/chemistry-organometallic-qa'
MODELO_HUB    = f'{HF_USERNAME}/{NOMBRE_MODELO}'

print(f'Modelo base    : {MODELO_BASE}')
print(f'Dataset        : {DATASET_ID}')
print(f'Modelo destino : {MODELO_HUB}')

HF_TOKEN cargado desde secretos de Colab.
Modelo base    : Qwen/Qwen2.5-1.5B-Instruct
Dataset        : Jesusrodriguezf90/chemistry-organometallic-qa
Modelo destino : Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora


## 2. Instalación de dependencias

In [6]:
# Instalación de dependencias para fine-tuning QLoRA
# transformers: carga y gestión del modelo base
# peft: implementación de LoRA (Low-Rank Adaptation)
# trl: SFTTrainer para Supervised Fine-Tuning
# bitsandbytes: quantización 4-bit para reducir uso de VRAM
# datasets: carga del dataset desde HF Hub
# accelerate: optimización del entrenamiento en GPU
!pip install transformers peft trl bitsandbytes datasets accelerate huggingface_hub --quiet
print('Dependencias instaladas.')

Dependencias instaladas.


In [7]:
# Imports de las librerías de fine-tuning
# Se importan tras la instalación para evitar ModuleNotFoundError
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from huggingface_hub import HfApi

print(f'torch version  : {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch version  : 2.10.0+cu128
CUDA disponible: True
VRAM total     : 15.6 GB


## 3. Carga del dataset y modelo base

In [8]:
# Cargar dataset desde HF Hub
# El dataset contiene 73 pares QA en formato messages (system/user/assistant)
# construidos en 05_dataset.ipynb sobre el paper PMC10967698
dataset = load_dataset(DATASET_ID, token=HF_TOKEN)
print(f'Dataset cargado: {DATASET_ID}')
print(f'Total ejemplos : {len(dataset["train"])}')
print(f'Columnas       : {dataset["train"].column_names}')
print(f'\nEjemplo par 0:')
ejemplo = dataset['train'][0]
for msg in ejemplo['messages']:
    print(f"  [{msg['role'].upper()}]: {msg['content'][:100]}...")

Dataset cargado: Jesusrodriguezf90/chemistry-organometallic-qa
Total ejemplos : 73
Columnas       : ['messages', 'fuente', 'calidad']

Ejemplo par 0:
  [SYSTEM]: You are a specialized scientific assistant in organometallic chemistry and medicinal inorganic chemi...
  [USER]: What metals are used in the NHC complexes studied?...
  [ASSISTANT]: The NHC complexes studied contain palladium (Pd), platinum (Pt) and gold (Au) as metal centers. Spec...


In [9]:
# Configuración de quantización 4-bit (QLoRA)
# Reduce el modelo de 32-bit a 4-bit para que quepa en la T4 (15GB VRAM)
# compute_dtype=float16: los cálculos internos se hacen en float16
# para mantener precisión numérica a pesar de la quantización
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',          # Normal Float 4 — mejor precisión que int4
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # Quantización doble — reduce VRAM adicional
)

print('Configuración QLoRA:')
print('  Quantización  : 4-bit NF4')
print('  Compute dtype : float16')
print('  Double quant  : True')

Configuración QLoRA:
  Quantización  : 4-bit NF4
  Compute dtype : float16
  Double quant  : True


In [10]:
# Cargar el tokenizador de Qwen2.5-1.5B
# Padding_side='right': necesario para SFTTrainer con causal LMs
tokenizer = AutoTokenizer.from_pretrained(
    MODELO_BASE,
    token=HF_TOKEN,
    padding_side='right',
)
tokenizer.pad_token = tokenizer.eos_token

print(f'Tokenizador cargado: {MODELO_BASE}')
print(f'Vocabulario        : {tokenizer.vocab_size:,} tokens')
print(f'Token de padding   : {tokenizer.pad_token}')

Tokenizador cargado: Qwen/Qwen2.5-1.5B-Instruct
Vocabulario        : 151,643 tokens
Token de padding   : <|im_end|>


In [11]:
# Cargar el modelo base Qwen2.5-1.5B en 4-bit
# device_map='auto': distribuye automáticamente las capas en GPU/CPU
# según la VRAM disponible
RUTA_MODELO_LOCAL = DIR_FINETUNE / 'modelo_base_cache'

# Si el modelo ya está en disco no se vuelve a descargar
# Qwen2.5-1.5B ocupa ~3GB — la descarga solo ocurre una vez
if RUTA_MODELO_LOCAL.exists():
    print(f'Modelo cargado desde caché local: {RUTA_MODELO_LOCAL}')
    origen = str(RUTA_MODELO_LOCAL)
else:
    print(f'Descargando modelo desde HF Hub: {MODELO_BASE}')
    origen = MODELO_BASE

modelo = AutoModelForCausalLM.from_pretrained(
    origen,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)

# Guardar en caché local para ejecuciones posteriores
if not RUTA_MODELO_LOCAL.exists():
    modelo.save_pretrained(str(RUTA_MODELO_LOCAL))
    tokenizer.save_pretrained(str(RUTA_MODELO_LOCAL))
    print(f'Modelo guardado en caché: {RUTA_MODELO_LOCAL}')

# Preparar el modelo para entrenamiento con gradientes de precisión mixta
modelo.config.use_cache = False   # Necesario durante el entrenamiento con LoRA
modelo.config.pretraining_tp = 1  # Tensor parallelism desactivado para GPU única

print(f'\nModelo cargado correctamente')
print(f'Parámetros totales: {sum(p.numel() for p in modelo.parameters()):,}')

Modelo cargado desde caché local: /content/drive/MyDrive/chem-rag-assistant/data/finetune/modelo_base_cache


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Modelo cargado correctamente
Parámetros totales: 888,616,448


## 4. Configuración LoRA y entrenamiento

In [12]:
# Configuración de los adaptadores LoRA
# LoRA (Low-Rank Adaptation) añade matrices pequeñas de rango bajo
# sobre las capas de atención del modelo. Solo estas matrices se entrenan
# dejando los pesos originales del modelo base completamente intactos.
#
# r=16: rango de las matrices LoRA — controla la capacidad de adaptación
#       valores más altos = más capacidad pero más VRAM
# lora_alpha=32: escala de los adaptadores (normalmente 2*r)
# target_modules: capas donde se aplican los adaptadores
#       en Qwen2.5 las capas de atención son q_proj, k_proj, v_proj, o_proj
# lora_dropout=0.05: regularización para evitar overfitting en datasets pequeños
# task_type=CAUSAL_LM: tipo de tarea — generación de texto autoregresivo
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

modelo_lora = get_peft_model(modelo, lora_config)

# Mostrar parámetros entrenables vs totales
# Con LoRA solo entrenamos ~1-3% de los parámetros totales
params_totales    = sum(p.numel() for p in modelo_lora.parameters())
params_entrenables = sum(p.numel() for p in modelo_lora.parameters() if p.requires_grad)
print(f'Parámetros totales     : {params_totales:,}')
print(f'Parámetros entrenables : {params_entrenables:,}')
print(f'Porcentaje entrenable  : {100 * params_entrenables / params_totales:.2f}%')

Parámetros totales     : 907,081,216
Parámetros entrenables : 18,464,768
Porcentaje entrenable  : 2.04%


In [13]:
# Configuración del entrenamiento
# num_train_epochs=5: 5 pasadas completas sobre el dataset de 73 pares
#   suficiente para un dataset pequeño sin riesgo alto de overfitting
# per_device_train_batch_size=2: 2 ejemplos por paso — equilibrio entre
#   velocidad y uso de VRAM en la T4
# gradient_accumulation_steps=4: acumula gradientes de 4 pasos antes de
#   actualizar pesos — simula un batch size efectivo de 8
# learning_rate=2e-4: tasa de aprendizaje estándar para QLoRA
# warmup_ratio=0.03: 3% de los pasos iniciales con lr creciente
#   evita inestabilidad al inicio del entrenamiento
# max_seq_length=512: longitud máxima de secuencia — suficiente para
#   nuestros pares QA de química
RUTA_OUTPUT = DIR_FINETUNE / 'checkpoints'

sft_config = SFTConfig(
    output_dir=str(RUTA_OUTPUT),
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    bf16=True,                          # Precisión mixta — reduce VRAM
    logging_steps=10,
    save_strategy='epoch',              # Guardar checkpoint al final de cada época
    save_total_limit=2,                 # Conservar solo los 2 últimos checkpoints
    max_length=512,
    packing=False,                      # Sin packing — dataset pequeño
    report_to='none',                   # Sin W&B ni MLflow en este prototipo
)

print('Configuración de entrenamiento:')
print(f'  Épocas          : {sft_config.num_train_epochs}')
print(f'  Batch size      : {sft_config.per_device_train_batch_size}')
print(f'  Grad. acum.     : {sft_config.gradient_accumulation_steps}')
print(f'  Batch efectivo  : {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}')
print(f'  Learning rate   : {sft_config.learning_rate}')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Configuración de entrenamiento:
  Épocas          : 5
  Batch size      : 2
  Grad. acum.     : 4
  Batch efectivo  : 8
  Learning rate   : 0.0002


In [14]:
# Inicializar el trainer y ejecutar el entrenamiento
# SFTTrainer aplica automáticamente el chat template de Qwen2.5
# a los mensajes en formato messages (system/user/assistant)
# evitando errores de tokenización por tokens incorrectos
trainer = SFTTrainer(
    model=modelo_lora,
    args=sft_config,
    train_dataset=dataset['train'],
    processing_class=tokenizer,
)
# Verificar si ya existe un checkpoint para reanudar
# Evita re-entrenar si el proceso se interrumpió
checkpoints = sorted(RUTA_OUTPUT.glob('checkpoint-*')) if RUTA_OUTPUT.exists() else []
resume_from = str(checkpoints[-1]) if checkpoints else None

if resume_from:
    print(f'Reanudando desde checkpoint: {resume_from}')
else:
    print('Iniciando entrenamiento desde cero...')

print(f'Total pasos estimados: {trainer.args.max_steps if trainer.args.max_steps > 0 else "auto"}')
print('Entrenando...')

resultado_entrenamiento = trainer.train(resume_from_checkpoint=resume_from)

print('\nEntrenamiento completado.')
print(f'Loss final : {resultado_entrenamiento.training_loss:.4f}')
print(f'Pasos      : {resultado_entrenamiento.global_step}')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Reanudando desde checkpoint: /content/drive/MyDrive/chem-rag-assistant/data/finetune/checkpoints/checkpoint-40
Total pasos estimados: auto
Entrenando...


Step,Training Loss
50,0.705448



Entrenamiento completado.
Loss final : 0.1411
Pasos      : 50


## 5. Guardado y publicación en HF Hub

In [15]:
# Guardar los adaptadores LoRA localmente
# Solo se guardan los adaptadores (~50MB) — no el modelo completo
# Los pesos originales de Qwen2.5-1.5B quedan intactos y se cargan
# por separado en inferencia usando PEFT
RUTA_ADAPTADORES = DIR_FINETUNE / 'lora_adapters'

if not RUTA_ADAPTADORES.exists():
    trainer.model.save_pretrained(str(RUTA_ADAPTADORES))
    tokenizer.save_pretrained(str(RUTA_ADAPTADORES))
    print(f'Adaptadores LoRA guardados en: {RUTA_ADAPTADORES}')
    tamanio = sum(f.stat().st_size for f in RUTA_ADAPTADORES.rglob('*') if f.is_file())
    print(f'Tamaño total: {tamanio / 1e6:.1f} MB')
else:
    print(f'Adaptadores ya guardados en: {RUTA_ADAPTADORES}')

Adaptadores LoRA guardados en: /content/drive/MyDrive/chem-rag-assistant/data/finetune/lora_adapters
Tamaño total: 85.3 MB


In [16]:
# Publicar los adaptadores LoRA en HuggingFace Hub
# Los adaptadores se publican como un modelo independiente en HF Hub
# Para usarlos se necesita cargar el modelo base + los adaptadores con PEFT
api = HfApi(token=HF_TOKEN)

# Crear el repositorio si no existe
try:
    api.create_repo(
        repo_id=MODELO_HUB,
        repo_type='model',
        exist_ok=True,
        private=False,
    )
    print(f'Repositorio: https://huggingface.co/{MODELO_HUB}')
except Exception as e:
    print(f'Repositorio ya existe o error: {e}')

# Subir los adaptadores
trainer.model.push_to_hub(
    MODELO_HUB,
    token=HF_TOKEN,
    commit_message='feat: adaptadores LoRA Qwen2.5-1.5B fine-tuned en química organometálica',
)
tokenizer.push_to_hub(
    MODELO_HUB,
    token=HF_TOKEN,
)

print(f'\nAdaptadores publicados: https://huggingface.co/{MODELO_HUB}')

Repositorio: https://huggingface.co/Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpq_evoajk/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.



Adaptadores publicados: https://huggingface.co/Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora


## 6. Resumen final

In [17]:
print('=' * 60)
print('RESUMEN — FINE-TUNING COMPLETADO')
print('=' * 60)
print(f'  Modelo base           : {MODELO_BASE}')
print(f'  Dataset               : {DATASET_ID}')
print(f'  Pares de entrenamiento: {len(dataset["train"])}')
print(f'  Épocas                : {sft_config.num_train_epochs}')
print(f'  Loss final            : {resultado_entrenamiento.training_loss:.4f}')
print(f'  Adaptadores LoRA      : https://huggingface.co/{MODELO_HUB}')
print('=' * 60)
print('Siguiente paso: 06b_evaluacion.ipynb — Evaluación comparativa')
print('  base vs fine-tuned sobre las 5 preguntas de ground truth')
print('=' * 60)

RESUMEN — FINE-TUNING COMPLETADO
  Modelo base           : Qwen/Qwen2.5-1.5B-Instruct
  Dataset               : Jesusrodriguezf90/chemistry-organometallic-qa
  Pares de entrenamiento: 73
  Épocas                : 5
  Loss final            : 0.1411
  Adaptadores LoRA      : https://huggingface.co/Jesusrodriguezf90/qwen2.5-1.5b-chemistry-lora
Siguiente paso: 06b_evaluacion.ipynb — Evaluación comparativa
  base vs fine-tuned sobre las 5 preguntas de ground truth
